# 01_CREDIT_OUTPUTS_v2_1

**Trương Minh Hoàng · MSSV 25410008**

Notebook nghiên cứu KLTN v2.1. Đọc [hướng dẫn chạy](README.md) và chuẩn bị dữ liệu trước khi thực thi. Output Colab cũ được bỏ; kết quả đã khóa nằm ở [evidence/historical](../evidence/historical/).

Research notebook: dataset-backed execution is separate from the small public demonstrations. Scientific feature formulas, model candidates and routing checks are retained. Public packaging changes cover paths, optional installation and fresh-run defaults.

# 01 — Sinh đầu ra tín dụng độc lập (v2.1)

Mục tiêu: tạo **Credit source outputs** cho Shared Risk Context. Đây không phải đóng góp mô hình tín dụng chính.

- Chia theo tháng `issue_d`: train ≥70%, validation đến ≥85% lũy tích, phần còn lại holdout; không tách một tháng.
- Nhãn holdout được lưu **tách riêng** khỏi file đầu vào cho RQ2.
- Score được gọi là `lightgbm_default_class_score`, không tuyên bố là PD ngân hàng đã hiệu chỉnh.


In [ ]:
# Install the optional research requirements before running.
# %pip -q install "lightgbm>=4.3,<5" pyarrow scikit-learn

In [ ]:
import os
from IPython.display import display
from pathlib import Path
import hashlib, json, os, random
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

POINTER_PATH = Path(os.environ.get('KLTN_CONFIG_POINTER', '/content/drive/MyDrive/KLTN_REPRODUCTION_CONFIG_POINTER.json' if Path('/content/drive/MyDrive').is_dir() else str(Path.cwd() / 'KLTN_REPRODUCTION_CONFIG_POINTER.json'))).resolve()
if not POINTER_PATH.exists():
    raise FileNotFoundError('Chưa có config pointer. Hãy chạy notebook 00 trước.')
CONFIG_PATH = Path(json.loads(POINTER_PATH.read_text(encoding='utf-8'))['config_path'])
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

ARTIFACT_DIR = Path(CONFIG['artifact_dir'])
CREDIT_DIR = Path(CONFIG['credit_dir'])
CREDIT_DIR.mkdir(parents=True, exist_ok=True)
LC_CSV = Path(CONFIG['lendingclub_csv'])
PRIMARY_SEED = int(CONFIG.get('primary_seed', 42))

random.seed(PRIMARY_SEED)
np.random.seed(PRIMARY_SEED)
print({'config': str(CONFIG_PATH), 'credit_dir': str(CREDIT_DIR), 'seed': PRIMARY_SEED})


In [ ]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

if not LC_CSV.exists():
    raise FileNotFoundError(f'Chưa có dữ liệu: {LC_CSV}')

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

DATASET_SHA256 = sha256_file(LC_CSV)
df = pd.read_csv(LC_CSV, low_memory=False)
required = {'id', 'issue_d', 'Default'}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f'Thiếu cột bắt buộc: {missing}')

df['_issue_month'] = pd.to_datetime(df['issue_d'], errors='coerce').dt.to_period('M')
if df['_issue_month'].isna().any():
    raise ValueError(f'Có {int(df["_issue_month"].isna().sum())} dòng không đọc được issue_d.')

default_numeric = pd.to_numeric(df['Default'], errors='coerce')
if default_numeric.isna().any():
    raise ValueError('Default có giá trị thiếu hoặc không đọc được; dừng để kiểm tra nguồn.')
if not set(default_numeric.unique()).issubset({0, 1}):
    raise ValueError('Default phải chỉ gồm 0 và 1.')
df['Default'] = default_numeric.astype('int8')

print({'rows': int(len(df)), 'columns': int(len(df.columns)), 'dataset_sha256': DATASET_SHA256})


In [ ]:
month_counts = df.groupby('_issue_month', observed=True).size().sort_index()
cumulative = month_counts.cumsum() / month_counts.sum()
train_end_month = cumulative[cumulative >= 0.70].index[0]
val_end_month = cumulative[cumulative >= 0.85].index[0]

train_mask = df['_issue_month'] <= train_end_month
val_mask = (df['_issue_month'] > train_end_month) & (df['_issue_month'] <= val_end_month)
holdout_mask = df['_issue_month'] > val_end_month

split_summary = pd.DataFrame([
    {'split': 'train', 'rows': int(train_mask.sum()),
     'month_min': str(df.loc[train_mask, '_issue_month'].min()), 'month_max': str(df.loc[train_mask, '_issue_month'].max())},
    {'split': 'validation', 'rows': int(val_mask.sum()),
     'month_min': str(df.loc[val_mask, '_issue_month'].min()), 'month_max': str(df.loc[val_mask, '_issue_month'].max())},
    {'split': 'holdout', 'rows': int(holdout_mask.sum()),
     'month_min': str(df.loc[holdout_mask, '_issue_month'].min()), 'month_max': str(df.loc[holdout_mask, '_issue_month'].max())},
])
if (split_summary['rows'] == 0).any():
    raise ValueError('Ít nhất một split rỗng; dừng trước huấn luyện.')
# Chỉ train/validation labels được kiểm tra trước khi model selection hoàn tất.
for split_name, mask in [('train', train_mask), ('validation', val_mask)]:
    vals = np.unique(df.loc[mask, 'Default'].to_numpy())
    if set(vals.tolist()) != {0, 1}:
        raise ValueError(f'{split_name} không có đủ hai lớp 0/1.')
display(split_summary)
split_summary.to_csv(CREDIT_DIR / 'credit_split_summary.csv', index=False)
print('Holdout label distribution: NOT INSPECTED before model freeze.')


In [ ]:
# Danh sách bảo thủ; manifest sẽ ghi chính xác những cột được dùng.
POST_OUTCOME_COLUMNS = {
    'loan_status', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d',
    'last_pymnt_amnt', 'next_pymnt_d', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'out_prncp', 'out_prncp_inv', 'last_credit_pull_d'
}
ALWAYS_EXCLUDE = {'id', 'issue_d', 'Default', '_issue_month'}
candidate_cols = [c for c in df.columns if c not in ALWAYS_EXCLUDE | POST_OUTCOME_COLUMNS]
train_df = df.loc[train_mask, candidate_cols]

numeric_cols = train_df.select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_cols = []
high_cardinality_excluded = []
for col in [c for c in candidate_cols if c not in numeric_cols]:
    n_unique = int(train_df[col].nunique(dropna=True))
    if n_unique <= 200:
        categorical_cols.append(col)
    else:
        high_cardinality_excluded.append(col)

feature_cols = numeric_cols + categorical_cols
if not feature_cols:
    raise ValueError('Không còn đặc trưng sau kiểm tra.')

category_maps = {}
for col in categorical_cols:
    values = train_df[col].fillna('__missing__').astype(str)
    category_maps[col] = {value: idx for idx, value in enumerate(sorted(values.unique()))}

def make_matrix(frame: pd.DataFrame) -> pd.DataFrame:
    output = pd.DataFrame(index=frame.index)
    for col in numeric_cols:
        output[col] = pd.to_numeric(frame[col], errors='coerce').astype('float32')
    for col in categorical_cols:
        output[col] = frame[col].fillna('__missing__').astype(str).map(category_maps[col]).fillna(-1).astype('int32')
    return output[feature_cols]

X_train = make_matrix(df.loc[train_mask])
X_val = make_matrix(df.loc[val_mask])
X_holdout = make_matrix(df.loc[holdout_mask])
y_train = df.loc[train_mask, 'Default'].to_numpy(dtype='int8')
y_val = df.loc[val_mask, 'Default'].to_numpy(dtype='int8')

feature_manifest = {
    'version': 'credit_features_v2.1',
    'numeric_features': numeric_cols,
    'categorical_features': categorical_cols,
    'post_outcome_excluded_present': sorted(set(df.columns) & POST_OUTCOME_COLUMNS),
    'high_cardinality_excluded': high_cardinality_excluded,
    'n_features': len(feature_cols),
}
(CREDIT_DIR / 'credit_feature_manifest.json').write_text(json.dumps(feature_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print({'numeric': len(numeric_cols), 'categorical': len(categorical_cols), 'total': len(feature_cols), 'high_cardinality_excluded': len(high_cardinality_excluded)})


In [ ]:
negative = int((y_train == 0).sum())
positive = int((y_train == 1).sum())
scale_pos_weight = negative / max(positive, 1)

candidates = [
    {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 50},
    {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 100},
    {'num_leaves': 31, 'learning_rate': 0.03, 'min_child_samples': 200},
]
tuning_rows, models = [], []
for candidate_id, candidate in enumerate(candidates):
    model = lgb.LGBMClassifier(
        objective='binary', n_estimators=3000, random_state=PRIMARY_SEED,
        n_jobs=-1, colsample_bytree=0.9, subsample=0.9, subsample_freq=1,
        reg_lambda=1.0, scale_pos_weight=scale_pos_weight,
        verbosity=-1, **candidate
    )
    model.fit(
        X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='average_precision',
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
    )
    val_score = model.predict_proba(X_val, num_iteration=model.best_iteration_)[:, 1]
    tuning_rows.append({
        'candidate_id': candidate_id,
        'validation_ap': float(average_precision_score(y_val, val_score)),
        'validation_roc_auc': float(roc_auc_score(y_val, val_score)),
        'best_iteration': int(model.best_iteration_),
        **candidate,
    })
    models.append(model)

tuning = pd.DataFrame(tuning_rows).sort_values(['validation_ap', 'candidate_id'], ascending=[False, True])
display(tuning)
tuning.to_csv(CREDIT_DIR / 'credit_validation_tuning.csv', index=False)
best_id = int(tuning.iloc[0]['candidate_id'])
model = models[best_id]


# Freeze credit source configuration BEFORE reading holdout labels.
MODEL_PATH = CREDIT_DIR / 'credit_lightgbm_v2.1.txt'
model.booster_.save_model(str(MODEL_PATH))
FEATURE_MANIFEST_PATH = CREDIT_DIR / 'credit_feature_manifest.json'
CREDIT_FREEZE_PATH = CREDIT_DIR / 'credit_source_freeze_manifest.json'
selected_row = tuning.iloc[0].to_dict()
credit_freeze = {
    'status': 'CREDIT_SOURCE_FREEZE_READY',
    'protocol_version': CONFIG['protocol_version'],
    'dataset_sha256': DATASET_SHA256,
    'split': {
        'train_end_month': str(train_end_month),
        'validation_end_month': str(val_end_month),
        'same_month_not_split': True,
    },
    'feature_manifest_sha256': sha256_file(FEATURE_MANIFEST_PATH),
    'candidate_grid': candidates,
    'selected_candidate': selected_row,
    'model_sha256': sha256_file(MODEL_PATH),
    'holdout_labels_accessed': False,
    'claim_boundary': 'independent credit source output; descriptive holdout evaluation; not calibrated bank PD',
}
CREDIT_FREEZE_PATH.write_text(json.dumps(credit_freeze, ensure_ascii=False, indent=2), encoding='utf-8')
print('CREDIT SOURCE FREEZE READY:', CREDIT_FREEZE_PATH)


In [ ]:
# Holdout labels are accessed only after the source-model configuration above has been frozen.
y_holdout = df.loc[holdout_mask, 'Default'].to_numpy(dtype='int8')
if set(np.unique(y_holdout).tolist()) != {0, 1}:
    raise ValueError('Credit holdout không có đủ hai lớp 0/1.')
holdout_score = model.predict_proba(X_holdout, num_iteration=model.best_iteration_)[:, 1]
metrics = {
    'model_version': 'credit_lightgbm_v2.1',
    'seed': PRIMARY_SEED,
    'holdout_rows': int(len(y_holdout)),
    'holdout_positive': int(y_holdout.sum()),
    'average_precision': float(average_precision_score(y_holdout, holdout_score)),
    'roc_auc': float(roc_auc_score(y_holdout, holdout_score)),
    'brier_score': float(brier_score_loss(y_holdout, holdout_score)),
    'train_end_month': str(train_end_month),
    'validation_end_month': str(val_end_month),
    'n_features': int(len(feature_cols)),
    'score_semantics': 'lightgbm_default_class_score',
    'claim_boundary': 'independent credit source output; not calibrated bank PD',
}

source_ref = df.loc[holdout_mask, 'id'].astype(str).to_numpy()
context_input = pd.DataFrame({
    'source_record_reference': source_ref,
    'entity_scope': 'lendingclub_loan_record',
    'risk_domain': 'credit',
    'score_value': holdout_score.astype('float64'),
    'score_semantics': 'lightgbm_default_class_score',
    'model_version': metrics['model_version'],
    'source_dataset': 'lendingclub_granting_models_v0.1',
    'assessment_time': df.loc[holdout_mask, 'issue_d'].astype(str).to_numpy(),
    'assessment_time_status': 'source_observed_month',
})
labels = pd.DataFrame({'source_record_reference': source_ref, 'y_true': y_holdout})

context_path = CREDIT_DIR / 'credit_predictions_for_context.parquet'
labels_path = CREDIT_DIR / 'credit_evaluation_labels.parquet'
metrics_path = CREDIT_DIR / 'credit_metrics.json'
context_input.to_parquet(context_path, index=False)
labels.to_parquet(labels_path, index=False)
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

manifest = {
    'status': 'CREDIT_SOURCE_READY',
    'freeze_manifest_sha256': sha256_file(CREDIT_FREEZE_PATH),
    'dataset_sha256': DATASET_SHA256,
    'files': {
        context_path.name: sha256_file(context_path),
        labels_path.name: sha256_file(labels_path),
        metrics_path.name: sha256_file(metrics_path),
        MODEL_PATH.name: sha256_file(MODEL_PATH),
        FEATURE_MANIFEST_PATH.name: sha256_file(FEATURE_MANIFEST_PATH),
    },
    'holdout_labels_accessed_after_freeze': True,
}
(CREDIT_DIR / 'credit_evidence_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('Đã lưu context input tách nhãn:', context_path.name, '+', labels_path.name)
print('CREDIT SOURCE READY')
